# Llama Sense Annotation Pipeline (Refactored)
This notebook uses the new modular pipeline for sense annotation with Llama, leveraging shared utilities for maintainability and traceability.

In [ ]:
from config import SENSE_REPO, ANNOTATIONS_TSV, OUTPUT_DIR
from process_senses import process_senses_with_chain
from writers import CustomWebAnnoTSVWriter, IncetprionWebAnnoTSVWriter
from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
import pandas as pd
import time

# Set model origin for traceability
ORIGIN_LLM = "Llama"

In [ ]:
# Load sense repository and annotated sentences
from data_loader import load_sense_repo
senses_df = load_sense_repo()
parser = WebAnnoLEXISParser(ANNOTATIONS_TSV)
sentences = parser.parse()

# Define test range (adjust as needed)
start_sentence = 0
end_sentence = start_sentence + 10  # For testing, use 10 sentences
sentences = sentences[start_sentence:end_sentence]

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Compose Llama LLM chain (do not change prompt_template)
prompt_template = PromptTemplate.from_template("""
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.
Odgovor mora biti isključivo u JSON formatu kao što je prikazano dole. Ne dodajete nikakav dodatni tekst pre ili posle odgovora.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Kontekst rečenice (ciljna reč je označena dvostrukim zvezdicama, npr. **reč**):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}

Ako nijedno značenje nije primenjivo ni približno, napišite "NEW_SENSE" kao `"sense_id"` 
i objasnite zašto nijedna opcija ne odgovara kontekstu.

Odgovor u isključivo sledećem JSON formatu:
{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko obrazloženje u jednoj ili dve rečenice>"
}}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
""")
llm = ChatOllama(
    model="llama3.3",
    temperature=0.0,
)
parser = StrOutputParser()
chain = prompt_template | llm | parser

In [ ]:
# Annotate sentences using the shared pipeline
start_time = time.time()
annotated_sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM
)
end_time = time.time()
print(f"Processed sentences {start_sentence} to {end_sentence} in {end_time - start_time:.2f} seconds.")

ENG30-01114303-v
U kontekstu dodele Kupa, 'osvojiti' znači savladati konkurenciju i zasluženo dobiti nagradu, što odgovara značenju 'savladati osvajanjem'.


In [ ]:
# Write debug output (all layers, including AI notes and candidates)
debug_writer = CustomWebAnnoTSVWriter(annotated_sentences)
debug_writer.save(OUTPUT_DIR / f"llama_debug_{start_sentence:04d}_{end_sentence:04d}.tsv")

Domains:
['sociology' 'psychiatry' 'factotum' 'chemistry' nan 'physiology'
 'literature' 'biology' 'medicine' 'administration' 'politics'
 'town_planning' 'geography' 'enterprise' 'person' 'psychology' 'zoology'
 'school' 'anthropology' 'linguistics' 'art' 'mechanics' 'play' 'exchange'
 'painting' 'law' 'grammar' 'building_industry' 'publishing' 'music'
 'mathematics' 'alimentation' 'metrology' 'pharmacy' 'physics' 'cinema'
 'philology' 'occultism' 'philosophy' 'quality' 'religion' 'architecture'
 'computer_science' 'military' 'geometry' 'astronomy' 'astronautics'
 'meteorology' 'tourism' 'transport' 'botany' 'banking' 'economy'
 'furniture' 'commerce' 'heraldry' 'atomic_physic' 'basketball'
 'free_time' 'cycling' 'sport' 'telephony' 'history' 'artisanship' 'color'
 'merchant_navy' 'boxing' 'mythology' 'gastronomy' 'number' 'anatomy'
 'money' 'time_period' 'entomology' 'hydraulics' 'dance' 'drawing'
 'optics' 'diplomacy' 'electricity' 'ethnology' 'industry' 'card'
 'betting' 'agricultu

In [ ]:
# Write Inception-compatible output (for annotation import)
incept_writer = IncetprionWebAnnoTSVWriter(annotated_sentences)
incept_writer.save(OUTPUT_DIR / f"llama_inception_{start_sentence:04d}_{end_sentence:04d}.tsv")

## Log processing time and completion

In [ ]:
with open("llama.log", "a", encoding="utf-8") as f:
    f.write(f"Processed sentences {start_sentence} to {end_sentence}\n")
    f.write(f"Llama took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")

MultiWordExpression(lemma='u skladu sa', token_count=3, token_indices=[6, 7, 8], type='AdpID', group_id='1')
MultiWordExpression(lemma='javljati se', token_count=2, token_indices=[8, 9], type='IRV', group_id='1')
MultiWordExpression(lemma='u znak', token_count=2, token_indices=[16, 17], type='AdpID', group_id='1')
